# Step 6 — Retrieval Adapters
**Spec refs:** SPEC-200 §5.4, REQ-200-071, REQ-200-ER1–ER5, REQ-200-160  
**Gap ref:** G-RETRIEVAL-01 — Director-directed architectural change 2026-05-10  

Two adapters, priority order:
1. `PubMedAdapter` — peer-reviewed literature via NCBI E-utilities
2. `WebSearchAdapter` — sanctioned-domain web scraping + external fallback

**Part A** — pure logic tests (no network).  
**Part B** — live network tests (require internet access).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import builtins
from unittest.mock import patch, MagicMock
print('Python:', sys.version)
print('Working dir:', os.getcwd())


Python: 3.11.15 | packaged by conda-forge | (main, Mar  5 2026, 16:36:00) [MSC v.1944 64 bit (AMD64)]
Working dir: D:\Git Repos\nikko-companion\notebooks


In [2]:
# Test 1: All public symbols import cleanly
from retrieval import PubMedAdapter, WebSearchAdapter, SANCTIONED_DOMAINS, ADAPTER_PRIORITY_ORDER
from retrieval.base_adapter import CachedBaseAdapter
from docs.schemas.retrieval_schemas import (
    RetrievalResult, RetrievalError, EvidenceItem,
    SourceTier, EvidenceTier,
)
print('PASS  All retrieval symbols import cleanly')


PASS  All retrieval symbols import cleanly


In [3]:
# Test 2: Sanctioned domain list matches the five Director-approved sources
EXPECTED = {
    'healthdirect.gov.au',
    'betterhealth.vic.gov.au',
    'who.int',
    'beyondblue.org.au',
    'blackdoginstitute.org.au',
}
assert set(SANCTIONED_DOMAINS) == EXPECTED, f'Domain mismatch. Got: {set(SANCTIONED_DOMAINS)}'
print(f'PASS  SANCTIONED_DOMAINS = {SANCTIONED_DOMAINS}')


PASS  SANCTIONED_DOMAINS = ['healthdirect.gov.au', 'betterhealth.vic.gov.au', 'who.int', 'beyondblue.org.au', 'blackdoginstitute.org.au']


In [4]:
# Test 3: Superseded adapters raise ImportError
import importlib
for mod_name in [
    'retrieval.healthdirect_adapter',
    'retrieval.better_health_adapter',
    'retrieval.who_adapter',
]:
    try:
        importlib.import_module(mod_name)
        print(f'FAIL  {mod_name} should have raised ImportError')
    except ImportError as e:
        print(f'PASS  {mod_name} -> ImportError: {e}')


PASS  retrieval.healthdirect_adapter -> ImportError: No module named 'retrieval.healthdirect_adapter'
PASS  retrieval.better_health_adapter -> ImportError: No module named 'retrieval.better_health_adapter'
PASS  retrieval.who_adapter -> ImportError: No module named 'retrieval.who_adapter'


In [5]:
# Test 4: Adapter priority order — REQ-200-071
assert ADAPTER_PRIORITY_ORDER[0] is PubMedAdapter, 'PubMed must be priority 1'
assert ADAPTER_PRIORITY_ORDER[1] is WebSearchAdapter, 'WebSearch must be priority 2'
assert len(ADAPTER_PRIORITY_ORDER) == 2
print(f'PASS  Priority order: {[a.__name__ for a in ADAPTER_PRIORITY_ORDER]}')


PASS  Priority order: ['PubMedAdapter', 'WebSearchAdapter']


In [6]:
# Test 5: WebSearchAdapter class-level constants
ws = WebSearchAdapter()
assert ws.SOURCE_TIER == SourceTier.PRIMARY
assert ws.EVIDENCE_TIER == EvidenceTier.GREY_LITERATURE
assert ws.CACHE_POLICY.ttl_days == 3
assert ws.CACHE_POLICY.head_check_interval is None
from retrieval.web_search_adapter import MIN_SANCTIONED_RESULTS
assert MIN_SANCTIONED_RESULTS >= 1
print(f'PASS  WebSearchAdapter constants OK')
print(f'      ttl_days={ws.CACHE_POLICY.ttl_days}, min_sanctioned={MIN_SANCTIONED_RESULTS}')


PASS  WebSearchAdapter constants OK
      ttl_days=3, min_sanctioned=2


In [7]:
# Test 6: _is_sanctioned correctly classifies domains
cases = [
    ('healthdirect.gov.au',      True),
    ('www.healthdirect.gov.au',  True),
    ('beyondblue.org.au',        True),
    ('blackdoginstitute.org.au', True),
    ('who.int',                  True),
    ('betterhealth.vic.gov.au',  True),
    ('reddit.com',               False),
    ('google.com',               False),
    ('psychologytoday.com',      False),
]
all_pass = True
for domain, expected in cases:
    got = ws._is_sanctioned(domain)
    ok = 'PASS' if got == expected else 'FAIL'
    if got != expected: all_pass = False
    print(f'  {ok}  _is_sanctioned({domain!r}) = {got}')
assert all_pass


  PASS  _is_sanctioned('healthdirect.gov.au') = True
  PASS  _is_sanctioned('www.healthdirect.gov.au') = True
  PASS  _is_sanctioned('beyondblue.org.au') = True
  PASS  _is_sanctioned('blackdoginstitute.org.au') = True
  PASS  _is_sanctioned('who.int') = True
  PASS  _is_sanctioned('betterhealth.vic.gov.au') = True
  PASS  _is_sanctioned('reddit.com') = False
  PASS  _is_sanctioned('google.com') = False
  PASS  _is_sanctioned('psychologytoday.com') = False


In [8]:
# Test 7: Cache key is deterministic and case-insensitive
key1 = ws._cache_key('anxiety management')
key2 = ws._cache_key('anxiety management')
key3 = ws._cache_key('ANXIETY MANAGEMENT')
assert key1 == key2, 'Cache key must be deterministic'
assert key1 == key3, 'Cache key must be case-insensitive'
assert len(key1) == 64, 'Cache key must be 64-char SHA-256 hex'
print(f'PASS  Cache key deterministic: {key1[:16]}...')


PASS  Cache key deterministic: 734abcf9ed650b5e...


In [9]:
# Test 8: DDG missing dependency -> RetrievalError (REQ-200-160)
# The adapter lazy-imports ddgs inside _ddg_search.
# Mocking 'ddgs' simulates it not being installed.
real_import = builtins.__import__

def mock_no_ddg(name, *args, **kwargs):
    if name == 'ddgs':  # matches: from ddgs import DDGS
        raise ImportError('mocked: not installed')
    return real_import(name, *args, **kwargs)

with patch('builtins.__import__', side_effect=mock_no_ddg):
    result = ws._ddg_search('anxiety', 5)

assert isinstance(result, RetrievalError), f'Expected RetrievalError, got {type(result).__name__}'
assert result.error_code == 'dependency_missing'
assert result.retryable is False
print('PASS  Missing dependency -> RetrievalError(dependency_missing, retryable=False)')

def mock_ddg_fail(name, *args, **kwargs):
    if name == 'ddgs':
        mod = MagicMock()
        mod.DDGS.return_value.__enter__.return_value.text.side_effect = RuntimeError('rate limited')
        return mod
    return real_import(name, *args, **kwargs)

with patch('builtins.__import__', side_effect=mock_ddg_fail):
    result2 = ws._ddg_search('anxiety', 5)

assert isinstance(result2, RetrievalError)
assert result2.error_code == 'search_error'
assert result2.retryable is True
print('PASS  DDG runtime error -> RetrievalError(search_error, retryable=True)')


PASS  Missing dependency -> RetrievalError(dependency_missing, retryable=False)
PASS  DDG runtime error -> RetrievalError(search_error, retryable=True)


In [10]:
# Test 9: PubMed XML parser — plain abstract
# EvidenceItem fields: title, abstract, url, source_name,
# publication_date, evidence_tier, source_tier, cache_hit, retrieved_at.
# No source_id or authors field. PMID lives in the url.
# source_name matches PubMedAdapter.SOURCE_NAME = 'PubMed Central Open Access'
from retrieval.pubmed_adapter import PubMedAdapter
from datetime import datetime, timezone

PLAIN_XML = '''
<PubmedArticleSet>
  <PubmedArticle>
    <MedlineCitation>
      <PMID>12345678</PMID>
      <Article>
        <ArticleTitle>CBT for anxiety disorders: a review</ArticleTitle>
        <Abstract>
          <AbstractText>This study reviewed CBT outcomes for anxiety.</AbstractText>
        </Abstract>
        <AuthorList>
          <Author><LastName>Smith</LastName><ForeName>J</ForeName></Author>
        </AuthorList>
        <Journal>
          <Title>Journal of Anxiety</Title>
          <JournalIssue>
            <PubDate><Year>2023</Year><Month>Jun</Month></PubDate>
          </JournalIssue>
        </Journal>
      </Article>
    </MedlineCitation>
  </PubmedArticle>
</PubmedArticleSet>
'''

pa = PubMedAdapter()
now = datetime.now(timezone.utc)
items = pa._parse_pubmed_xml(PLAIN_XML, retrieved_at=now)

assert len(items) == 1
item = items[0]
assert '12345678' in item.url, f'PMID missing from URL: {item.url}'
assert 'CBT' in item.title, f'Title mismatch: {item.title}'
assert 'CBT outcomes' in item.abstract, f'Abstract mismatch: {item.abstract}'
assert item.source_name == 'PubMed Central Open Access', f'source_name: {item.source_name}'
assert item.publication_date is not None, 'publication_date should not be None'
print('PASS  PubMed XML parser — plain abstract')
print(f'      url={item.url}')
print(f'      title={item.title[:55]}')
print(f'      pub_date={item.publication_date}')


PASS  PubMed XML parser — plain abstract
      url=https://pubmed.ncbi.nlm.nih.gov/12345678/
      title=CBT for anxiety disorders: a review
      pub_date=2023-Jun


In [11]:
# Test 10: External fallback results carry scrutiny prefix (REQ-200-ER5)
from retrieval.web_search_adapter import _EXTERNAL_SCRUTINY_PREFIX
from datetime import datetime, timezone

assert _EXTERNAL_SCRUTINY_PREFIX
assert 'EXTERNAL' in _EXTERNAL_SCRUTINY_PREFIX.upper()
print(f'PASS  Scrutiny prefix exists: {_EXTERNAL_SCRUTINY_PREFIX[:60]}...')

mock_ddg_result = [{
    'title': 'Anxiety management tips',
    'href':  'https://psychologytoday.com/anxiety',
    'body':  'Some advice about managing anxiety.',
}]
with patch.object(ws, '_ddg_search', return_value=mock_ddg_result):
    now = datetime.now(timezone.utc)
    items = ws._phase2_external('anxiety', max_results=3, now=now)

assert not isinstance(items, RetrievalError), f'Expected list, got error: {items}'
assert len(items) == 1
ext = items[0]
assert ext.source_tier == SourceTier.SECONDARY, f'Expected SECONDARY, got {ext.source_tier}'
assert _EXTERNAL_SCRUTINY_PREFIX in ext.abstract, f'Prefix missing: {ext.abstract[:80]}'
print('PASS  External items carry SourceTier.SECONDARY + scrutiny prefix')


PASS  Scrutiny prefix exists: [EXTERNAL — HIGHER SCRUTINY REQUIRED: this result is from a ...
PASS  External items carry SourceTier.SECONDARY + scrutiny prefix


---
## Part B — Live Network Tests
These cells make real HTTP requests. Skip if offline — each cell has a built-in SKIP path.


In [12]:
# [PART B] DDG sanctioned domain search
ws_live = WebSearchAdapter()
raw = ws_live._ddg_search(
    '(site:beyondblue.org.au OR site:healthdirect.gov.au) anxiety depression',
    max_results=5
)
if isinstance(raw, RetrievalError):
    print(f'SKIP  DDG unavailable: {raw.error_message}')
else:
    print(f'DDG returned {len(raw)} results')
    for r in raw:
        domain = ws_live._extract_domain(r.get('href', ''))
        marker = 'SANCTIONED' if ws_live._is_sanctioned(domain) else 'EXTERNAL'
        print(f'  [{marker}] {domain}: {r.get("title", "")[:60]}')
    sanctioned_count = sum(
        1 for r in raw
        if ws_live._is_sanctioned(ws_live._extract_domain(r.get('href', '')))
    )
    print(f'Sanctioned: {sanctioned_count}/{len(raw)}')
    assert len(raw) > 0
    print('PASS  DDG live search returned results')


DDG returned 5 results
  [SANCTIONED] beyondblue.org.au: 24/7 Support for Anxiety, Depression and Suicide Prevention.
  [SANCTIONED] beyondblue.org.au: Anxiety and Depression Test K10 - Beyond Blue
  [SANCTIONED] beyondblue.org.au: Understand Anxiety disorders - Beyond Blue
  [SANCTIONED] healthdirect.gov.au: Anxiety and Depression Program | healthdirect
  [SANCTIONED] healthdirect.gov.au: Depression - symptoms, types, treatment | healthdirect
Sanctioned: 5/5
PASS  DDG live search returned results


In [13]:
# [PART B] Content scraping from a sanctioned domain
import re
content = ws_live._scrape_content('https://www.beyondblue.org.au/mental-health/anxiety')
if content is None:
    print('SKIP  Scraping unavailable')
else:
    has_html = bool(re.search(r'<[a-z]+[^>]*>', content, re.IGNORECASE))
    assert len(content) >= 100, f'Too short: {len(content)} chars'
    assert not has_html, 'Raw HTML tags in scraped content'
    print(f'PASS  Beyond Blue scraped {len(content)} chars of clean text')
    print(f'Preview: {content[:200]}')


PASS  Beyond Blue scraped 404 chars of clean text
Preview: Loading component... Loading component... Loading component... Loading component... Loading component... Loading component... Related information Loading component... References Beyond Blue uses stati


In [14]:
# [PART B] Full WebSearchAdapter.search() end-to-end
# WebSearchAdapter.search() takes StaticCacheQueryParams (the generic query container)
from docs.schemas.retrieval_schemas import StaticCacheQueryParams
params = StaticCacheQueryParams(query='depression support Australia', max_results=4)
result = ws_live.search(params)
if isinstance(result, RetrievalError):
    print(f'SKIP  search() error: {result.error_message}')
else:
    assert isinstance(result, RetrievalResult)
    print(f'Items: {len(result.items)}')
    for item in result.items:
        tier = 'PRIMARY' if item.source_tier == SourceTier.PRIMARY else 'SECONDARY'
        print(f'  [{tier}] {item.url} | {item.title[:50]}')
    print('PASS  WebSearchAdapter.search() returned valid RetrievalResult')


Items: 4
  [PRIMARY] https://www.healthdirect.gov.au/mental-health-helplines | Mental health helplines | healthdirect
  [PRIMARY] https://www.beyondblue.org.au/ | 24/7 Support for Anxiety, Depression and Suicide P
  [PRIMARY] https://www.blackdoginstitute.org.au/resources-support/support-groups/ | Support groups for depression, anxiety & bipolar -
  [PRIMARY] https://www.betterhealth.vic.gov.au/health/conditionsandtreatments/depression-treatment-and-management | Depression - treatment and management | Better Hea
PASS  WebSearchAdapter.search() returned valid RetrievalResult


In [15]:
# [PART B] PubMed E-utilities live call
from docs.schemas.retrieval_schemas import PubMedQueryParams
pm = PubMedAdapter()
pm_params = PubMedQueryParams(query='cognitive behavioural therapy anxiety', max_results=3)
pm_result = pm.search(pm_params)
if isinstance(pm_result, RetrievalError):
    print(f'SKIP  PubMed unavailable: {pm_result.error_message}')
else:
    assert isinstance(pm_result, RetrievalResult)
    print(f'PubMed returned {len(pm_result.items)} items')
    for item in pm_result.items:
        print(f'  {item.url}')
        print(f'  {item.title[:65]}')
        print(f'  Abstract: {item.abstract[:100]}...')
    if pm_result.items:
        assert pm_result.items[0].abstract
    print('PASS  PubMed E-utilities live call succeeded')


PubMed returned 3 items
  https://pubmed.ncbi.nlm.nih.gov/42094328/
  Comparative effectiveness of non-pharmacological interventions fo
  Abstract: BACKGROUND: Social anxiety disorder (SAD) is a prevalent and disabling mental health condition in ad...
  https://pubmed.ncbi.nlm.nih.gov/42066645/
  Digital health interventions for improving mental health outcomes
  Abstract: AIMS: Digital interventions have emerged as promising tools to support mental well-being in diabetes...
  https://pubmed.ncbi.nlm.nih.gov/42058068/
  Effects of mindfulness yoga intervention on physical function, an
  Abstract: BACKGROUND: Methamphetamine (MA) addiction is a chronic relapsing disorder that impairs physical hea...
PASS  PubMed E-utilities live call succeeded
